# Train Bilingual Conformer ASR in Google Colab

Run the cells from top to bottom. Before starting, choose **Runtime → Change runtime type → GPU**. This notebook keeps model outputs in Google Drive so checkpoints can survive a Colab disconnect.

**Dataset required:** the course audio and manifests are not stored in this GitHub repository. Delta's `/work/nvme/bjdh/shared/coding4` directory is not available from Colab. Put the course-provided `dump` directory in `MyDrive/coding4/dump` (it should include `raw/train_bilingual/data.json`, `raw/dev_bilingual/data.json`, and referenced audio). Follow the course's data access and sharing rules.

The Delta `sbatch` script is not used here. Training runs directly through `train.py`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_DIR = Path('/content/Bilingual-Conformer-ASR')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Yen-hub/Bilingual-Conformer-ASR.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', Path.cwd())

In [ ]:
# Install Python packages without replacing Colab's CUDA-enabled PyTorch.
%pip install -r requirements.txt

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Colab choose Runtime → Change runtime type → GPU, reconnect, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this if you placed the dataset folder somewhere else in MyDrive.
DATA_DUMP = Path('/content/drive/MyDrive/coding4/dump')
required_manifests = [
    DATA_DUMP / 'raw/train_bilingual/data.json',
    DATA_DUMP / 'raw/dev_bilingual/data.json',
]
missing = [str(p) for p in required_manifests if not p.is_file()]
if missing:
    raise FileNotFoundError('Course dataset not found. Place the course-provided dump folder at ' + str(DATA_DUMP) + '. Missing: ' + ', '.join(missing))

repo_dump = REPO_DIR / 'dump'
if repo_dump.is_symlink():
    repo_dump.unlink()
elif repo_dump.exists():
    raise FileExistsError(f'{repo_dump} exists; move it before linking the Drive dataset.')
repo_dump.symlink_to(DATA_DUMP, target_is_directory=True)
print('Dataset linked:', repo_dump, '→', DATA_DUMP)

## Train

The recipe configuration sets the model and epoch count. The output directory below is on Drive. If Colab disconnects after at least one checkpoint has been saved, set `RESUME_FROM` to that checkpoint path and rerun this cell. The first run can take a while; its epoch timing will help estimate the total runtime and compute-unit use.

In [ ]:
OUTPUT_ROOT = Path('/content/drive/MyDrive/coding4_colab_results')
RESUME_FROM = ''  # Optional: full path to a saved .pth checkpoint
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

train_command = [
    sys.executable, 'train.py',
    '--tag', 'multilingual',
    '--mode', 'multilingual',
    '--train_json', 'dump/raw/train_bilingual/data.json',
    '--valid_json', 'dump/raw/dev_bilingual/data.json',
    '--out_dir', str(OUTPUT_ROOT),
    '--nworkers', '2',
]
if RESUME_FROM.strip():
    train_command += ['--resume', RESUME_FROM.strip()]
print('Running:', ' '.join(train_command))
subprocess.run(train_command, cwd=REPO_DIR, check=True)

## Find the saved checkpoint

The best checkpoint is saved under `MyDrive/coding4_colab_results/exp/train_multilingual/ckpts`. It can be supplied as `RESUME_FROM` if training needs to continue in a later session. Keep the Delta job queued separately if you still want it to run there.

In [ ]:
CHECKPOINT_DIR = OUTPUT_ROOT / 'exp/train_multilingual/ckpts'
checkpoints = sorted(CHECKPOINT_DIR.glob('*.pth')) if CHECKPOINT_DIR.exists() else []
for path in checkpoints:
    print(path)
if not checkpoints:
    print('No checkpoint saved yet.')